# 🧪 实验三：训练流程与代码解读

**MobileNetV3 图像分类实战教程**

---

## 实验目标

1. 理解完整的训练流程（数据加载 → 模型初始化 → 训练循环 → 验证 → 保存）
2. 深入学习关键训练技巧：余弦退火学习率、标签平滑、EMA
3. 掌握训练日志的解读方法
4. 理解分布式训练的基本概念

---


In [ ]:
# ====== 1. 导入所需库 ======
import torch
import torch.nn as nn
import torch.optim as optim
from copy import deepcopy
import math
import os

# 导入 torch_npu（Ascend NPU 支持）
import torch_npu

print(f"PyTorch 版本: {torch.__version__}")

import warnings
warnings.filterwarnings("ignore", message="Glyph.*missing from font")
warnings.filterwarnings("ignore", message=".*owner does not match.*")
warnings.filterwarnings("ignore", message=".*TASK_QUEUE_ENABLE.*")
warnings.filterwarnings("ignore", message=".*Permission mismatch.*")

In [ ]:
# ====== MobileNetV3 完整模型定义（自包含，不依赖外部文件）======

def get_model_parameters(model):
    total_parameters = 0
    for layer in list(model.parameters()):
        layer_parameter = 1
        for l in list(layer.size()):
            layer_parameter *= l
        total_parameters += layer_parameter
    return total_parameters

def _make_divisible(v, divisor=8, min_value=None):
    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v

def _weights_init(m):
    if isinstance(m, nn.Conv2d):
        torch.nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            torch.nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm2d):
        m.weight.data.fill_(1)
        m.bias.data.zero_()
    elif isinstance(m, nn.Linear):
        n = m.weight.size(1)
        m.weight.data.normal_(0, 0.01)
        m.bias.data.zero_()

class h_sigmoid(nn.Module):
    def __init__(self, inplace=True):
        super().__init__()
        self.inplace = inplace
    def forward(self, x):
        return F.relu6(x + 3., inplace=self.inplace) / 6.

class h_swish(nn.Module):
    def __init__(self, inplace=True):
        super().__init__()
        self.inplace = inplace
    def forward(self, x):
        out = F.relu6(x + 3., self.inplace) / 6.
        return out * x

class SqueezeBlock(nn.Module):
    def __init__(self, exp_size, divide=4):
        super().__init__()
        self.dense = nn.Sequential(
            nn.Linear(exp_size, exp_size // divide),
            nn.ReLU(inplace=True),
            nn.Linear(exp_size // divide, exp_size),
            h_sigmoid()
        )
    def forward(self, x):
        batch, channels, height, width = x.size()
        out = F.avg_pool2d(x, kernel_size=[height, width]).view(batch, -1)
        out = self.dense(out).view(batch, channels, 1, 1)
        return out * x

class MobileBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernal_size, stride, nonLinear, SE, exp_size):
        super().__init__()
        padding = (kernal_size - 1) // 2
        self.use_connect = stride == 1 and in_channels == out_channels
        self.SE = SE
        activation = nn.ReLU if nonLinear == "RE" else h_swish
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, exp_size, 1, 1, 0, bias=False),
            nn.BatchNorm2d(exp_size), activation(inplace=True))
        self.depth_conv = nn.Sequential(
            nn.Conv2d(exp_size, exp_size, kernal_size, stride, padding, groups=exp_size),
            nn.BatchNorm2d(exp_size))
        if self.SE:
            self.squeeze_block = SqueezeBlock(exp_size)
        self.point_conv = nn.Sequential(
            nn.Conv2d(exp_size, out_channels, 1, 1, 0),
            nn.BatchNorm2d(out_channels), activation(inplace=True))
    def forward(self, x):
        out = self.depth_conv(self.conv(x))
        if self.SE:
            out = self.squeeze_block(out)
        out = self.point_conv(out)
        return x + out if self.use_connect else out

class MobileNetV3(nn.Module):
    def __init__(self, model_mode="LARGE", num_classes=1000, multiplier=1.0, dropout_rate=0.0):
        super().__init__()
        self.num_classes = num_classes
        if model_mode == "LARGE":
            layers = [
                [16, 16, 3, 1, "RE", False, 16], [16, 24, 3, 2, "RE", False, 64],
                [24, 24, 3, 1, "RE", False, 72], [24, 40, 5, 2, "RE", True, 72],
                [40, 40, 5, 1, "RE", True, 120], [40, 40, 5, 1, "RE", True, 120],
                [40, 80, 3, 2, "HS", False, 240], [80, 80, 3, 1, "HS", False, 200],
                [80, 80, 3, 1, "HS", False, 184], [80, 80, 3, 1, "HS", False, 184],
                [80, 112, 3, 1, "HS", True, 480], [112, 112, 3, 1, "HS", True, 672],
                [112, 160, 5, 1, "HS", True, 672], [160, 160, 5, 2, "HS", True, 672],
                [160, 160, 5, 1, "HS", True, 960],
            ]
            init_conv_out = _make_divisible(16 * multiplier)
            self.init_conv = nn.Sequential(
                nn.Conv2d(3, init_conv_out, 3, 2, 1), nn.BatchNorm2d(init_conv_out), h_swish())
            self.block = nn.Sequential(*[
                MobileBlock(_make_divisible(ic * multiplier), _make_divisible(oc * multiplier),
                           k, s, nl, se, _make_divisible(exp * multiplier))
                for ic, oc, k, s, nl, se, exp in layers])
            self.out_conv1 = nn.Sequential(
                nn.Conv2d(_make_divisible(160 * multiplier), _make_divisible(960 * multiplier), 1, 1),
                nn.BatchNorm2d(_make_divisible(960 * multiplier)), h_swish())
            self.out_conv2 = nn.Sequential(
                nn.Conv2d(_make_divisible(960 * multiplier), _make_divisible(1280 * multiplier), 1, 1),
                h_swish(), nn.Dropout(dropout_rate),
                nn.Conv2d(_make_divisible(1280 * multiplier), num_classes, 1, 1))
        elif model_mode == "SMALL":
            layers = [
                [16, 16, 3, 2, "RE", True, 16], [16, 24, 3, 2, "RE", False, 72],
                [24, 24, 3, 1, "RE", False, 88], [24, 40, 5, 2, "RE", True, 96],
                [40, 40, 5, 1, "RE", True, 240], [40, 40, 5, 1, "RE", True, 240],
                [40, 48, 5, 1, "HS", True, 120], [48, 48, 5, 1, "HS", True, 144],
                [48, 96, 5, 2, "HS", True, 288], [96, 96, 5, 1, "HS", True, 576],
                [96, 96, 5, 1, "HS", True, 576],
            ]
            init_conv_out = _make_divisible(16 * multiplier)
            self.init_conv = nn.Sequential(
                nn.Conv2d(3, init_conv_out, 3, 2, 1), nn.BatchNorm2d(init_conv_out), h_swish())
            self.block = nn.Sequential(*[
                MobileBlock(_make_divisible(ic * multiplier), _make_divisible(oc * multiplier),
                           k, s, nl, se, _make_divisible(exp * multiplier))
                for ic, oc, k, s, nl, se, exp in layers])
            self.out_conv1 = nn.Sequential(
                nn.Conv2d(_make_divisible(96 * multiplier), _make_divisible(576 * multiplier), 1, 1),
                SqueezeBlock(_make_divisible(576 * multiplier)),
                nn.BatchNorm2d(_make_divisible(576 * multiplier)), h_swish())
            self.out_conv2 = nn.Sequential(
                nn.Conv2d(_make_divisible(576 * multiplier), _make_divisible(1280 * multiplier), 1, 1),
                h_swish(), nn.Dropout(dropout_rate),
                nn.Conv2d(_make_divisible(1280 * multiplier), num_classes, 1, 1))
        self.apply(_weights_init)
    def forward(self, x):
        out = self.block(self.init_conv(x))
        out = self.out_conv1(out)
        b, c, h, w = out.size()
        return self.out_conv2(F.avg_pool2d(out, [h, w])).view(b, -1)

print("✅ MobileNetV3 模型定义加载完成")

---
## 2. 训练流程概览

`main.py` 中的训练流程如下：

```
1. 解析参数 (get_args)
2. 初始化分布式 (init_distributed)
3. 加载数据 (load_data → DataLoader)
4. 创建模型 (MobileNetV3)
5. 初始化 EMA
6. 加载 checkpoint（可选）
7. 设置优化器 (RMSprop) + 学习率调度 (CosineLR)
8. 设置损失函数 (LabelSmoothingCrossEntropy)
9. 训练循环（每个 epoch）：
   a. 更新学习率
   b. 训练一个 epoch
   c. 验证一个 epoch
   d. 保存最佳模型
   e. 记录日志
10. 绘制训练曲线
```

---
## 3. 余弦退火学习率 (Cosine Annealing LR)

本项目的 `CosineLR` 实现了带 warmup 的余弦退火学习率调度：

**Warmup 阶段（前 5 个 epoch）：**
- 学习率从 $10^{-6}$ 线性上升到初始学习率
- 防止模型在初始阶段因为过大的学习率而不稳定

**余弦退火阶段：**
- 学习率从初始值平滑下降到 $10^{-5}$
- 公式：$\text{lr} = \text{min\_lr} + 0.5 \times (\text{base\_lr} - \text{min\_lr}) \times (1 + \cos(\pi \times \text{progress}))$

In [ ]:
# ====== 2. 实现并可视化 CosineLR ======

class CosineLR:
    """带 warmup 的余弦退火学习率调度"""
    def __init__(self, optimizer, total_epochs=200, warmup_epochs=5, warmup_lr_init=1e-6, min_lr=1e-5):
        self.optimizer = optimizer
        self.total_epochs = total_epochs
        self.warmup_epochs = warmup_epochs
        self.warmup_lr_init = warmup_lr_init
        self.min_lr = min_lr
        self.base_lr = optimizer.param_groups[0]['lr']

    def step(self, epoch):
        """epoch 从 1 开始"""
        if epoch <= self.warmup_epochs:
            lr = self.warmup_lr_init + (self.base_lr - self.warmup_lr_init) * epoch / self.warmup_epochs
        else:
            progress = (epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + math.cos(math.pi * progress))
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
        return lr


# 可视化学习率变化
import matplotlib.pyplot as plt
import numpy as np

# 模拟优化器
dummy_model = nn.Linear(10, 10)
optimizer = optim.SGD(dummy_model.parameters(), lr=0.01)

# 创建调度器
scheduler = CosineLR(optimizer, total_epochs=200, warmup_epochs=5, warmup_lr_init=1e-6, min_lr=1e-5)

# 记录学习率
lrs = []
for epoch in range(1, 201):
    lr = scheduler.step(epoch)
    lrs.append(lr)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, 201), lrs, 'b-', linewidth=2)
plt.axvline(x=5, color='r', linestyle='--', alpha=0.7, label='Warmup end')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.title('Cosine Annealing LR (0.01 → 1e-5)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(range(1, 15), lrs[:14], 'b-o', linewidth=2, markersize=4)
plt.axvline(x=5.5, color='r', linestyle='--', alpha=0.7, label='Warmup end')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.title('First 15 epochs (Warmup)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 Warmup 阶段：学习率从 1e-6 线性上升到 0.01（前 5 epoch）")
print("💡 余弦退火阶段：学习率平滑下降到 1e-5")
print("💡 相比 StepLR 的阶梯下降，余弦退火更平滑，有助于收敛到更好的局部最优")

In [ ]:
# ====== matplotlib 中文显示配置 ======
import matplotlib.pyplot as plt
import matplotlib
import warnings

# 尝试配置中文字体，如果不可用则回退到英文
try:
    import subprocess
    result = subprocess.run(['fc-list', ':lang=zh'], capture_output=True, text=True, timeout=3)
    if result.stdout.strip():
        font_name = result.stdout.split('\n')[0].split(':')[1].strip().split(',')[0]
        plt.rcParams['font.sans-serif'] = [font_name, 'DejaVu Sans']
    else:
        plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
except Exception:
    plt.rcParams['font.sans-serif'] = ['DejaVu Sans']

plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore', message='Glyph.*missing from font')
print("✅ matplotlib 显示配置完成")


---
## 4. 标签平滑 (Label Smoothing)

传统的交叉熵损失使用 one-hot 标签（正确类别为 1，其余为 0），这可能导致模型过自信。

**标签平滑**软化目标分布：

$$\text{target}_i = \begin{cases} 1 - \varepsilon & i = \text{正确类别} \\ \frac{\varepsilon}{N-1} & i \neq \text{正确类别} \end{cases}$$

其中 $\varepsilon=0.1$ 是平滑因子，$N$ 是类别数。

In [ ]:
# ====== 3. 标签平滑损失函数 ======

class LabelSmoothingCrossEntropy(nn.Module):
    """标签平滑交叉熵损失"""
    def __init__(self, smoothing=0.1):
        super(LabelSmoothingCrossEntropy, self).__init__()
        self.smoothing = smoothing

    def forward(self, x, target):
        log_probs = torch.nn.functional.log_softmax(x, dim=-1)
        n_classes = x.size(-1)
        with torch.no_grad():
            # 构建平滑后的目标分布
            true_dist = torch.zeros_like(log_probs)
            true_dist.fill_(self.smoothing / (n_classes - 1))
            true_dist.scatter_(1, target.data.unsqueeze(1), 1.0 - self.smoothing)
        return torch.mean(torch.sum(-true_dist * log_probs, dim=-1))


# 演示标签平滑的效果
criterion_smooth = LabelSmoothingCrossEntropy(smoothing=0.1)
criterion_standard = nn.CrossEntropyLoss()

# 模拟 logits 和标签
logits = torch.randn(4, 200)  # 4个样本, 200类
targets = torch.randint(0, 200, (4,))

loss_smooth = criterion_smooth(logits, targets)
loss_standard = criterion_standard(logits, targets)

print(f"标准交叉熵损失: {loss_standard.item():.4f}")
print(f"标签平滑交叉熵: {loss_smooth.item():.4f}")

# 查看平滑后的目标分布
sample_logits = torch.randn(1, 5)
sample_target = torch.tensor([2])  # 假设正确类别是索引2
log_probs = torch.nn.functional.log_softmax(sample_logits, dim=-1)
n_classes = 5
true_dist = torch.zeros_like(log_probs)
true_dist.fill_(0.1 / (n_classes - 1))
true_dist.scatter_(1, sample_target.data.unsqueeze(1), 1.0 - 0.1)

print("\n标签平滑目标分布示例（5类，平滑因子=0.1，正确类别=索引2）:")
for i in range(5):
    print(f"  类别 {i}: {true_dist[0, i].item():.4f}")
print(f"💡 正确类别概率从 1.0 降低到 {1-0.1:.1f}")
print(f"💡 错误类别概率从 0.0 提升到 {0.1/4:.4f}")
print(f"💡 这可以防止模型过自信，提高泛化能力")

---
## 5. 指数移动平均 (EMA)

EMA 在训练过程中维护模型参数的滑动平均：

$$\theta_{\text{EMA}}^{(t)} = \alpha \cdot \theta_{\text{EMA}}^{(t-1)} + (1 - \alpha) \cdot \theta^{(t)}$$

其中 $\alpha = 0.9999$。EMA 模型通常比直接训练的模型有更好的泛化性能。

In [ ]:
# ====== 4. 实现并理解 EMA ======

class EMA:
    """指数移动平均"""
    def __init__(self, model, decay=0.9999):
        self.model = deepcopy(model).eval()
        self.model.requires_grad_(False)
        self.decay = decay

    def update_parameters(self, model):
        """在每个训练 step 后更新 EMA 参数"""
        with torch.no_grad():
            for ema_param, param in zip(self.model.parameters(), model.parameters()):
                ema_param.data.mul_(self.decay).add_(param.data, alpha=1 - self.decay)


# 演示 EMA 更新过程
demo_model = nn.Linear(2, 1)
ema_model = EMA(demo_model, decay=0.9)  # 使用 0.9 方便观察

print("EMA 更新演示 (decay=0.9):")
print(f"  初始 EMA 权重: {ema_model.model.weight.data.numpy().flatten()}")
print(f"  初始模型权重: {demo_model.weight.data.numpy().flatten()}")

for step in range(5):
    # 模拟模型权重更新
    with torch.no_grad():
        demo_model.weight.data += 0.1  # 模拟梯度更新
    
    # 更新 EMA
    ema_model.update_parameters(demo_model)
    
    print(f"  Step {step+1}: 模型={demo_model.weight.data.numpy().flatten()}, "
          f"EMA={ema_model.model.weight.data.numpy().flatten()}")

print("\n💡 EMA 参数的变化比模型参数更平滑，具有记忆效果")
print("💡 实际训练中 decay=0.9999，EMA 的变化非常缓慢")

---
## 6. 权重衰减 (Weight Decay)

本项目的 `add_weight_decay` 函数实现了**选择性权重衰减**：
- 对 Conv2d 和 Linear 的权重参数应用 weight decay
- 对 bias 和 BatchNorm 的参数**不**应用 weight decay

这是因为 BatchNorm 的缩放参数本身已经起到了正则化的作用，再应用 weight decay 反而会干扰训练。

In [ ]:
# ====== 5. 选择性权重衰减 ======

def add_weight_decay(model, weight_decay=1e-5):
    """将 weight decay 只应用到 Conv 和 Linear 的 weight，不应用到 bias 和 bn"""
    decay = []
    no_decay = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if 'bias' in name or 'bn' in name or 'norm' in name:
            no_decay.append(param)
        else:
            decay.append(param)
    return [
        {'params': no_decay, 'weight_decay': 0.},
        {'params': decay, 'weight_decay': weight_decay}
    ]


# 在模型上测试
model = MobileNetV3(model_mode="LARGE", num_classes=200)
param_groups = add_weight_decay(model, weight_decay=1e-5)

n_decay = len(param_groups[0]['params'])
n_no_decay = len(param_groups[1]['params'])
total = n_decay + n_no_decay

print(f"参数分组统计:")
print(f"  应用 weight decay 的参数数: {n_decay}")
print(f"  不应用 weight decay 的参数数: {n_no_decay}")
print(f"  总参数数: {total}")
print(f"\n💡 不对 bias 和 BN 参数做 weight decay，可以避免干扰归一化层的学习")

---
## 7. 分布式训练简介

`main.py` 支持使用 `torchrun` 进行多卡分布式训练：

```bash
torchrun --nproc_per_node=4 main.py --dataset-mode TINY_IMAGENET --distributed
```

关键点：
- 使用 `DistributedDataParallel` (DDP)
- 在 NPU 环境下，分布式后端使用 `hccl` 替代 `nccl`
- 学习率自动乘以设备数量
- 验证结果通过 `all_reduce` 在各设备间同步
- 只有 rank 0 负责日志记录和模型保存


In [ ]:
# ====== 7. 检查可用 NPU ======
import torch
import torch_npu

if torch.npu.is_available():
    print(f"可用 NPU 数量: {torch.npu.device_count()}")
    for i in range(torch.npu.device_count()):
        print(f"  NPU {i}: {torch.npu.get_device_name(i)}")
else:
    print("未检测到 NPU，将使用 CPU 训练（速度会很慢）")
    print("💡 请在华为昇腾平台上运行本教程")

---
## 📝 实验小结

在本实验中，我们深入解读了 `main.py` 训练流程的每个关键组件：

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">组件</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">作用</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">关键参数</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>CosineLR</strong></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">余弦退火学习率 + Warmup</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">warmup=5, min_lr=1e-5</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>LabelSmoothingCrossEntropy</strong></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">标签平滑，防止过自信</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">smoothing=0.1</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>EMA</strong></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">指数移动平均，提升泛化</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">decay=0.9999</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>add_weight_decay</strong></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">选择性权重衰减</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">decay=1e-5</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>RMSprop</strong></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">优化器（原文推荐）</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">alpha=0.9, eps=1e-3</td>
    </tr>
  </tbody>
</table>

这些技巧组合使用，使得 MobileNetV3-Large 在 Tiny ImageNet 上达到了 **61.47%** 的 Top-1 准确率。

---


## 课后练习

1. (单选题) CosineAnnealingLR(T_max=50, eta_min=0) 中 T_max 的含义是？
   - A. 完成半个余弦周期的 epoch 数
   - B. 总训练 epoch 数
   - C. 初始学习率
   - D. 最小学习率

2. (单选题) 200 类分类任务中，标签平滑 epsilon=0.1 时，错误类别的目标值是多少？
   - A. 0.1/199
   - B. 0.9/199
   - C. 0.1/200
   - D. 0

3. (单选题) EMA 更新公式 shadow = decay*shadow + (1-decay)*param 中，decay=0.999 的作用是？
   - A. 更看重近期参数
   - B. 更看重长期历史参数
   - C. 与历史无关
   - D. 等价于学习率

4. (多选题) 将 batch_size 从 64 提升到 256，通常需要同步调整？
   - A. 学习率
   - B. warmup 步数
   - C. 梯度裁剪阈值
   - D. 数据增强策略

5. (多选题) 一个规范的 Warmup + Cosine 学习率计划包含？
   - A. 线性/常数 warmup 阶段
   - B. 余弦衰减阶段
   - C. 最小学习率
   - D. 总训练步数

6. (判断题) RMSprop 通过除以梯度平方的滑动平均来归一化每个参数的学习步长。

7. (判断题) EMA 影子参数应在 optimizer.step() 之前更新，才能包含本轮梯度。

8. (填空题) model.eval() 会关闭 ____ 层，并让 BatchNorm 使用 ____ 统计量。

9. (填空题) 在昇腾 NPU 上使用混合精度训练时，防止梯度下溢通常使用 ____ 类做梯度缩放。

10. (简答题) 标签平滑为什么能抑制模型过度自信？请从 softmax 输出和目标分布角度说明。

11. (简答题) 为什么线性 warmup 能降低大学习率带来的早期不稳定性？

12. (代码设计题) 写出 train_one_step 骨架：forward → loss → zero_grad → backward → 可选 grad clip → step → EMA 更新。

13. (单选题) 训练 loss 持续下降，验证准确率连续 8 个 epoch 不提升，最可能的判断是？
   - A. 过拟合
   - B. 欠拟合
   - C. 学习率过高
   - D. 数据标签错误

14. (多选题) 训练日志出现以下哪些信号时，应暂停训练并排查稳定性？
   - A. loss 出现 NaN
   - B. 梯度范数指数级增大
   - C. 验证 loss 剧烈震荡
   - D. 训练 acc 单调上升

15. (简答题) 设计一个 CPU/NPU 公平对比实验，说明需要固定的变量与测量方法。

> 参考答案见 answer/02.05_training_pipeline_answer.ipynb。